Eksik Veri Analizi

In [ ]:
print("\n--- Eksik Değerler ---")
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "MissingCount": missing,
    "MissingPercent": missing_pct
})

missing_df

# Eksik değerlerin sayısı ve yüzdesi
plt.figure(figsize=(10,6))
sns.heatmap(df.isnull(), cbar=False)
plt.title("Eksik Veri Isı Haritası")
plt.show()

Temel Betimsel İstatistikler (Ortalama, Sapma, vb.)

In [ ]:
print("--- Sayısal Değişkenlerin Betimsel İstatistikleri ---")

numeric_cols = df.select_dtypes(include=np.number).columns

df[numeric_cols].hist(figsize=(14, 10), bins=30)
plt.suptitle("Sayısal Değişkenlerin Dağılımı", y=1.02)
plt.show()



# .T (transpoze) tabloyu daha okunabilir kılar
desc_stats = df.describe().T
print(desc_stats[['count', 'mean', 'std', 'min', '50%', 'max']])

# Özellikle odaklanacağımız değişkenlerin özetini ayrıca vurgulayalım
print("\n--- Odak Değişkenler İçin Özet (ProfitMargin & ESG_Overall) ---")
print(df[['ProfitMargin', 'ESG_Overall']].describe().T[['count', 'mean', 'std', 'min', 'max']])





Korelasyon Analizi

In [ ]:

# İlgili tüm nümerik özellikleri seçelim
corr_features = [
    'GrowthRate', 'Revenue', 'ProfitMargin', 'MarketCap', 
    'ESG_Overall', 'ESG_Environmental', 'ESG_Social', 'ESG_Governance', 
    'CarbonEmissions', 'WaterUsage', 'EnergyConsumption'
]

# Veri setinde var olan sütunlarla filtrele
valid_features = [col for col in corr_features if col in df.columns]

# Korelasyon matrisini oluştur
corr_matrix = df[valid_features].corr()

# Sadece 'GrowthRate' sütununu seç, kendini (1.00) listeden çıkar
growth_correlations = corr_matrix['GrowthRate'].drop('GrowthRate')

# Net saptama için korelasyonları en güçlüden en zayıfa sırala
sorted_growth_corrs = growth_correlations.sort_values(ascending=False)

print("\n'GrowthRate' Sütununun Diğer Özellikler ile Korelasyonu:")
print("(İyileştirilmiş veriye göre sıralanmıştır)")
print("-" * 50)
print(sorted_growth_corrs.to_string()) # .to_string() ile daha güzel format
print("-" * 50)


# --- 4. Görselleştirme (Bar Grafiği) ---
plt.figure(figsize=(10, 7))
# 'vlag' paleti pozitifler için kırmızı, negatifler için mavi tonlarını verir
sns.barplot(x=sorted_growth_corrs.values, y=sorted_growth_corrs.index, palette='vlag')
plt.title("'GrowthRate' ile Diğer Özelliklerin Korelasyonu\n(Şirket Medyanı ile Doldurulmuş Veri)", fontsize=14)
plt.xlabel("Korelasyon Katsayısı (r)")
plt.ylabel("Özellikler (Features)")
# 0 noktasını belirginleştirmek için bir çizgi ekle
plt.axvline(x=0, color='black', linewidth=0.8, linestyle='--')
plt.tight_layout()
plt.savefig('growthrate_correlations.png')

print("\nKorelasyon grafiği 'growthrate_correlations.png' olarak kaydedildi.")

EKSİK VERİ DOLDURMA

In [ ]:


# ----------------------------------------------------------------------
# NaN değerlerini doldurma işlemi
# ----------------------------------------------------------------------

# 'CompanyID' sütununa göre gruplama yapılır.
# transform() metodu ile her bir şirketin GrowthRate medyanı hesaplanır.
# fillna() ile NaN değerler bu medyan ile doldurulur.
df['GrowthRate'] = df.groupby('CompanyID')['GrowthRate'].transform(lambda x: x.fillna(x.median()))

# ----------------------------------------------------------------------

# Sonucu kontrol etmek için CompanyID 1'in verilerine bakabilirsiniz
# print(df[df['CompanyID'] == 1][['CompanyID', 'Year', 'GrowthRate']].sort_values(by='Year'))


#KONTROL

print("\n--- Eksik Değerler ---")
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "MissingCount": missing,
    "MissingPercent": missing_pct
})

missing_df

# Eksik değerlerin sayısı ve yüzdesi
plt.figure(figsize=(10,6))
sns.heatmap(df.isnull(), cbar=False)
plt.title("Eksik Veri Isı Haritası")
plt.show()

print("--- Veri Seti Genel Bilgileri ---")
print("Shape:", df.shape)
print("\nInfo:")
df.info()

print("\nDescribe:")
df.describe(include="all")
